In [4]:
#importing necessary libraries
from pathlib import Path
import numpy as np
import pandas as pd
import mne
from scipy.io import loadmat
from sklearn.decomposition import PCA

##  Processing individual recordings

The `process_recording()` function converts one raw recording and its
corresponding event file into an MNE-Python `Epochs` object.

For each recording, the function performs the following steps:

1. **Load the raw MATLAB data**

   The MATLAB file contains the three spatial components of the signal
   (`x`, `y`, and `z`) for each MEG node, together with the recording
   time vector.

2. **Combine the three spatial components**

   For each MEG node, the three components are combined using
   principal component analysis (PCA).

   For each node, the signal is represented as:

   `x, y, z → PCA → first principal component`

   The first principal component provides a single time series capturing
   the dominant variance across the three spatial components.

   Nodes containing NaN values are skipped.

3. **Load the event information**

   The corresponding `.evt` file contains the event onset times and
   trigger codes.

   Event onsets are taken from the `Tsec` column, while event identities
   are taken from the `TriNo` column.

4. **Create an MNE Raw object**

   The PCA-reduced signals are converted into an MNE `RawArray`, with
   one channel for each MEG node.

5. **Map experimental events**

   The original trigger codes are mapped to four event classes:

   | Trigger | Event |
   |---------|-------|
   | 9       | stay left |
   | 13      | stay right |
   | 17      | shift left |
   | 21      | shift right |

6. **Extract epochs**

   Continuous data are segmented around each event from **−0.5 to
   +1.0 seconds**.

   Baseline correction is performed using the **−0.5 to 0 second**
   pre-event interval.

The function returns an MNE `Epochs` object containing the processed
trials for that recording.

In [5]:
#Main function to process the recordings and extract epochs
# =========================================================
# PROCESSING FUNCTION 
# =========================================================
def process_recording(mat_file, evt_file,
                      tmin=-0.5,
                      tmax=1,
                      baseline=(-0.5, 0)):

    data = loadmat(mat_file)

    x = data['sigx_tot'].squeeze()
    y = data['sigy_tot'].squeeze()
    z = data['sigz_tot'].squeeze()
    time = data['time_ica'].squeeze()

    n_nodes = x.shape[0]
    n_timepoints = x.shape[1]

    combined_signal = np.zeros((n_nodes, n_timepoints))

    for i in range(n_nodes):
        data_node = np.vstack([x[i], y[i], z[i]]).T

        if np.isnan(data_node).any():
            continue

        pca = PCA(n_components=1)
        combined_signal[i] = pca.fit_transform(data_node)[:, 0]

    triggers = pd.read_csv(evt_file, sep=r"\s+", engine="python")

    onsets = triggers.Tsec.values
    duration = np.ones(len(onsets))
    descr = triggers.TriNo.astype(str).values

    annotations = mne.Annotations(
        onset=onsets,
        duration=duration,
        description=descr
    )

    dt = np.median(np.diff(time))
    sfreq = 1 / dt

    info = mne.create_info(
        ch_names=[f"node{i}" for i in range(n_nodes)],
        sfreq=sfreq,
        ch_types='misc'
    )

    raw = mne.io.RawArray(combined_signal, info)
    raw.set_annotations(annotations)

    # =====================================================
    # FIXED EVENT MAPPING
    # =====================================================
    FIXED_EVENT_ID = {
        "9": 1,
        "13": 2,
        "17": 3,
        "21": 4
    }
#     "stay_left": 9,
#     "stay_right": 13,
#     "shift_left": 17,
#     "shift_right": 21,
    events, event_id = mne.events_from_annotations(
        raw,
        event_id=FIXED_EVENT_ID
    )

    epochs = mne.Epochs(
        raw,
        events,
        event_id=FIXED_EVENT_ID,
        tmin=tmin,
        tmax=tmax,
        baseline=baseline,
        preload=True
    )

    return epochs

# MEG Data Preprocessing and Epoching

## 1. Overview

This notebook contains the preprocessing pipeline used to convert the
raw MEG recordings into epoched MNE-Python datasets for the subsequent
decoding and replay analyses.

The raw data are organized by participant. For each participant, the
pipeline identifies matching `.mat` and `.evt` files corresponding to
individual recording runs.

Each matched MAT/EVT pair is processed using the `process_recording()`
function. The resulting epochs from all runs belonging to the same
participant are then concatenated into a single dataset.

The final epoched datasets are saved in FIF format and assigned
sequential subject identifiers (`Sub1`, `Sub2`, etc.).

### Processing steps

For each participant, the pipeline:

1. Identifies the participant folder.
2. Finds all `.mat` and `.evt` recording files.
3. Matches MAT and EVT files based on their filename.
4. Processes each matched recording using `process_recording()`.
5. Concatenates all runs from the same participant.
6. Saves the resulting epochs as an MNE `.fif` file.

The processing output printed below provides a record of which
participants and recording runs were successfully processed and
whether any recordings were skipped.

In [6]:
# =========================================================
# PIPELINE 
# =========================================================

base_path = Path("/home/uranus/Scrivania/MEG_replay/data")
#output_path = Path("/home/uranus/MEG_replay/epoched_data")
#output_path.mkdir(parents=True, exist_ok=True)

participant_dirs = sorted([p for p in base_path.iterdir() if p.is_dir()])

sub_counter = 1

for participant_path in participant_dirs:

    participant_id = participant_path.name

    print(f"\n==============================")
    print(f"Processing {participant_id}")
    print(f"==============================")

    mat_files = sorted(participant_path.glob("*.mat"))
    evt_files = sorted(participant_path.glob("*.evt"))

    mat_dict = {f.stem: f for f in mat_files}
    evt_dict = {f.stem: f for f in evt_files}

    common_keys = sorted(set(mat_dict.keys()) & set(evt_dict.keys()))

    if len(common_keys) == 0:
        print("No matching MAT/EVT pairs found, skipping.")
        continue

    participant_epochs = []

    for key in common_keys:

        print(f"  Run: {key}")

        epochs = process_recording(
            mat_file=str(mat_dict[key]),
            evt_file=str(evt_dict[key])
        )

        participant_epochs.append(epochs)

    if len(participant_epochs) == 0:
        continue

    # merge runs
    epochs_all = mne.concatenate_epochs(participant_epochs)

    # save with Sub numbering
    #save_name = f"Sub{sub_counter}_epo.fif"
    #save_path = output_path / save_name

    #epochs_all.save(save_path, overwrite=True)

    #print(f"Saved: {save_name}")

    sub_counter += 1
    
    


Processing bnrcld87_01
  Run: bnrcld87_0108
Creating RawArray with float64 data, n_channels=99, n_times=507607
    Range : 0 ... 507606 =      0.000 ...   990.451 secs
Ready.
Used Annotations descriptions: [np.str_('13'), np.str_('17'), np.str_('21'), np.str_('9')]
Not setting metadata
180 matching events found
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 180 events and 770 original time points ...
0 bad epochs dropped
  Run: bnrcld87_0112


/tmp/ipykernel_1409653/265660254.py:53: RuntimeWarning: Omitted 1 annotation(s) that were outside data range.
  raw.set_annotations(annotations)


Creating RawArray with float64 data, n_channels=99, n_times=506097
    Range : 0 ... 506096 =      0.000 ...   987.504 secs
Ready.
Used Annotations descriptions: [np.str_('13'), np.str_('17'), np.str_('21'), np.str_('9')]
Not setting metadata
183 matching events found
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 183 events and 770 original time points ...
0 bad epochs dropped
Not setting metadata
363 matching events found
Applying baseline correction (mode: mean)


/tmp/ipykernel_1409653/265660254.py:53: RuntimeWarning: Omitted 1 annotation(s) that were outside data range.
  raw.set_annotations(annotations)
/tmp/ipykernel_1409653/2218796134.py:50: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  epochs_all = mne.concatenate_epochs(participant_epochs)



Processing cllmgh87_03
  Run: cllmgh87_0308
Creating RawArray with float64 data, n_channels=99, n_times=503090
    Range : 0 ... 503089 =      0.000 ...   981.637 secs
Ready.
Used Annotations descriptions: [np.str_('13'), np.str_('17'), np.str_('21'), np.str_('9')]
Not setting metadata
183 matching events found
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 183 events and 770 original time points ...


...


Processing zcclra92_01
  Run: zcclra92_0110
Creating RawArray with float64 data, n_channels=99, n_times=504426
    Range : 0 ... 504425 =      0.000 ...   984.244 secs
Ready.
Used Annotations descriptions: [np.str_('13'), np.str_('17'), np.str_('21'), np.str_('9')]
Not setting metadata
183 matching events found
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 183 events and 770 original time points ...
0 bad epochs dropped
  Run: zcclra92_0114


/tmp/ipykernel_1409653/265660254.py:53: RuntimeWarning: Omitted 1 annotation(s) that were outside data range.
  raw.set_annotations(annotations)


Creating RawArray with float64 data, n_channels=99, n_times=499665
    Range : 0 ... 499664 =      0.000 ...   974.954 secs
Ready.
Used Annotations descriptions: [np.str_('13'), np.str_('17'), np.str_('21'), np.str_('9')]
Not setting metadata
180 matching events found
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 180 events and 770 original time points ...
0 bad epochs dropped


/tmp/ipykernel_1409653/265660254.py:53: RuntimeWarning: Omitted 1 annotation(s) that were outside data range.
  raw.set_annotations(annotations)
/tmp/ipykernel_1409653/2218796134.py:50: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  epochs_all = mne.concatenate_epochs(participant_epochs)


Not setting metadata
363 matching events found
Applying baseline correction (mode: mean)
